# NiyamTrace-X Q1 Experiment 3 — Verified Execution Broker, Replay, TOCTOU, and Atomicity

**Goal:** experimentally close the execution-boundary limitation acknowledged by the paper.

The reference broker binds authorization to actor, tool, argument hash, predicted targets,
database snapshot, policy hash, expiry and nonce. It tests replay, stale-state, argument
tampering, policy drift, actor/tool mismatch, expiry, signature forgery, delta mismatch
defense, and atomic batch rollback.

This notebook is CPU-only and deterministic.

In [ ]:
# Reproducible setup: pin the exact public repository commit audited in the manuscript.
REPO_URL = "https://github.com/bnssaanirudh/NiyamTrace-X.git"
PINNED_COMMIT = "c14661dbd11c42ebd1019b6a1a5c49b8643da137"

!rm -rf /content/NiyamTrace-X
!git clone -q $REPO_URL /content/NiyamTrace-X
%cd /content/NiyamTrace-X
!git checkout -q $PINNED_COMMIT

!pip -q install -e /content/NiyamTrace-X/niyamtrace
!pip -q install pandas numpy scipy scikit-learn matplotlib tqdm statsmodels nbformat

from pathlib import Path
import os, json, math, random, hashlib, statistics, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/NiyamTrace-X")
NIYAM = ROOT / "niyamtrace"
RESULTS = Path("/content/niyamtrace_q1_results")
RESULTS.mkdir(exist_ok=True)
print("Pinned commit:", PINNED_COMMIT)
print("Results:", RESULTS)

In [ ]:
import sqlite3, hmac, hashlib, json, secrets, time
from dataclasses import dataclass, asdict

def canonical(obj):
    return json.dumps(obj,sort_keys=True,separators=(",",":"),default=str)

def sha(obj):
    return hashlib.sha256(canonical(obj).encode()).hexdigest()

def fresh_db():
    conn=sqlite3.connect(":memory:")
    conn.row_factory=sqlite3.Row
    conn.execute("""
        CREATE TABLE vendor_invoices(
            invoice_id TEXT PRIMARY KEY,
            vendor_id INTEGER NOT NULL,
            month INTEGER NOT NULL,
            year INTEGER NOT NULL,
            status TEXT NOT NULL
        )
    """)
    conn.executemany(
        "INSERT INTO vendor_invoices VALUES(?,?,?,?,?)",
        [
            ("I1",4421,3,2025,"OPEN"),
            ("I2",4421,3,2025,"OPEN"),
            ("I3",4421,3,2025,"OPEN"),
            ("I4",4421,4,2025,"OPEN"),
            ("I5",7777,3,2025,"OPEN"),
        ]
    )
    conn.commit()
    return conn

def snapshot_hash(conn):
    return hashlib.sha256("\n".join(conn.iterdump()).encode()).hexdigest()

def predict_archive(conn,args):
    rows=conn.execute(
        """SELECT invoice_id FROM vendor_invoices
           WHERE vendor_id=? AND month=? AND year=? AND status='OPEN'
           ORDER BY invoice_id""",
        (int(args["vendor_id"]),int(args["month"]),int(args["year"]))
    ).fetchall()
    return [r["invoice_id"] for r in rows]

def current_status(conn):
    return {r["invoice_id"]:r["status"] for r in conn.execute(
        "SELECT invoice_id,status FROM vendor_invoices ORDER BY invoice_id"
    )}

@dataclass
class Capability:
    actor:str
    tool:str
    args_hash:str
    predicted_ids_hash:str
    snapshot_hash:str
    policy_hash:str
    expires_at:float
    nonce:str
    signature:str=""

    def unsigned(self):
        d=asdict(self)
        d.pop("signature",None)
        return d

class VerifiedExecutionBroker:
    def __init__(self,conn,secret_key=None,policy_hash="POLICY-v1"):
        self.conn=conn
        self.secret_key=secret_key or secrets.token_bytes(32)
        self.policy_hash=policy_hash
        self.used_nonces=set()

    def _sign(self,payload):
        return hmac.new(self.secret_key,canonical(payload).encode(),hashlib.sha256).hexdigest()

    def issue(self,actor,tool,args,ttl_seconds=60):
        if tool!="archive_invoices":
            raise ValueError("reference broker supports archive_invoices")
        predicted=predict_archive(self.conn,args)
        cap=Capability(
            actor=actor,tool=tool,args_hash=sha(args),
            predicted_ids_hash=sha(predicted),snapshot_hash=snapshot_hash(self.conn),
            policy_hash=self.policy_hash,expires_at=time.time()+ttl_seconds,
            nonce=secrets.token_hex(16)
        )
        cap.signature=self._sign(cap.unsigned())
        return cap

    def verify(self,cap,actor,tool,args):
        if not hmac.compare_digest(cap.signature,self._sign(cap.unsigned())):
            return False,"BAD_SIGNATURE"
        if cap.nonce in self.used_nonces:
            return False,"REPLAY"
        if time.time()>cap.expires_at:
            return False,"EXPIRED"
        if actor!=cap.actor:
            return False,"ACTOR_MISMATCH"
        if tool!=cap.tool:
            return False,"TOOL_MISMATCH"
        if sha(args)!=cap.args_hash:
            return False,"ARGS_TAMPERED"
        if self.policy_hash!=cap.policy_hash:
            return False,"POLICY_CHANGED"
        if snapshot_hash(self.conn)!=cap.snapshot_hash:
            return False,"STALE_SNAPSHOT"
        if sha(predict_archive(self.conn,args))!=cap.predicted_ids_hash:
            return False,"PREDICTION_CHANGED"
        return True,"OK"

    def commit(self,cap,actor,tool,args):
        ok,reason=self.verify(cap,actor,tool,args)
        if not ok:
            return {"committed":False,"reason":reason,"actual_ids":[]}
        predicted=predict_archive(self.conn,args)
        before=current_status(self.conn)
        try:
            self.conn.execute("BEGIN IMMEDIATE")
            self.conn.execute(
                """UPDATE vendor_invoices SET status='ARCHIVED'
                   WHERE vendor_id=? AND month=? AND year=? AND status='OPEN'""",
                (int(args["vendor_id"]),int(args["month"]),int(args["year"]))
            )
            after=current_status(self.conn)
            actual=sorted(k for k,v in after.items() if before.get(k)!=v and v=="ARCHIVED")
            if actual!=sorted(predicted):
                self.conn.rollback()
                return {"committed":False,"reason":"ACTUAL_DELTA_MISMATCH","actual_ids":actual}
            self.conn.commit()
            self.used_nonces.add(cap.nonce)
            return {"committed":True,"reason":"OK","actual_ids":actual}
        except Exception as e:
            self.conn.rollback()
            return {"committed":False,"reason":f"EXECUTION_ERROR:{type(e).__name__}","actual_ids":[]}

print("Reference broker defined.")

In [ ]:
BASE_ARGS={"vendor_id":4421,"month":3,"year":2025}
ACTOR="procurement_manager"

def run_attack(name):
    conn=fresh_db()
    broker=VerifiedExecutionBroker(conn)
    cap=broker.issue(ACTOR,"archive_invoices",BASE_ARGS,ttl_seconds=60)

    if name=="valid":
        out=broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS)); expected=True
    elif name=="replay":
        broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS))
        out=broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS)); expected=False
    elif name=="args_tamper":
        bad=dict(BASE_ARGS); bad["month"]=4
        out=broker.commit(cap,ACTOR,"archive_invoices",bad); expected=False
    elif name=="stale_snapshot":
        conn.execute("UPDATE vendor_invoices SET status='PAID' WHERE invoice_id='I1'"); conn.commit()
        out=broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS)); expected=False
    elif name=="policy_change":
        broker.policy_hash="POLICY-v2"
        out=broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS)); expected=False
    elif name=="expired":
        cap.expires_at=time.time()-1
        cap.signature=broker._sign(cap.unsigned())
        out=broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS)); expected=False
    elif name=="wrong_actor":
        out=broker.commit(cap,"unauthorized","archive_invoices",dict(BASE_ARGS)); expected=False
    elif name=="wrong_tool":
        out=broker.commit(cap,ACTOR,"delete_invoices",dict(BASE_ARGS)); expected=False
    elif name=="forged_signature":
        cap.signature="00"*32
        out=broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS)); expected=False
    elif name=="predicted_ids_tamper":
        cap.predicted_ids_hash=sha(["I1"])
        out=broker.commit(cap,ACTOR,"archive_invoices",dict(BASE_ARGS)); expected=False
    else:
        raise ValueError(name)

    return {"attack":name,"expected_commit":expected,**out,"pass":bool(out["committed"])==expected}

ATTACKS=[
    "valid","replay","args_tamper","stale_snapshot","policy_change",
    "expired","wrong_actor","wrong_tool","forged_signature","predicted_ids_tamper"
]
attack_df=pd.DataFrame([run_attack(a) for a in ATTACKS])
display(attack_df)
assert attack_df["pass"].all()

In [ ]:
# Demonstrate the legacy architectural pattern: raw execution can mutate without a capability.
def legacy_direct_execute(conn,args):
    before=current_status(conn)
    conn.execute(
        """UPDATE vendor_invoices SET status='ARCHIVED'
           WHERE vendor_id=? AND month=? AND year=? AND status='OPEN'""",
        (args["vendor_id"],args["month"],args["year"])
    )
    conn.commit()
    after=current_status(conn)
    return sorted(k for k,v in after.items() if before[k]!=v)

conn=fresh_db()
legacy_changed=legacy_direct_execute(conn,BASE_ARGS)
print("Legacy direct invocation changed:",legacy_changed)
assert legacy_changed==["I1","I2","I3"]

In [ ]:
TRIALS_PER_ATTACK=250
trial_rows=[]
for attack in ATTACKS:
    for i in range(TRIALS_PER_ATTACK):
        rec=run_attack(attack)
        rec["trial"]=i
        trial_rows.append(rec)

trials=pd.DataFrame(trial_rows)
metrics=trials.groupby("attack").agg(
    n=("pass","size"),success_rate=("pass","mean"),committed_rate=("committed","mean")
).reset_index()

security={
    "n_trials":int(len(trials)),
    "all_expected_behaviors":bool(trials["pass"].all()),
    "attack_block_rate_excluding_valid":float(1-trials[trials.attack!="valid"]["committed"].mean()),
    "valid_commit_rate":float(trials[trials.attack=="valid"]["committed"].mean()),
}
display(metrics)
print(json.dumps(security,indent=2))

In [ ]:
# Atomic batch reference design: one capability binds the whole operation list.
@dataclass
class BatchCapability:
    actor:str
    operations_hash:str
    snapshot_hash:str
    policy_hash:str
    expires_at:float
    nonce:str
    signature:str=""

    def unsigned(self):
        d=asdict(self); d.pop("signature",None); return d

class AtomicBatchBroker(VerifiedExecutionBroker):
    def issue_batch(self,actor,operations,ttl_seconds=60):
        cap=BatchCapability(
            actor=actor,operations_hash=sha(operations),snapshot_hash=snapshot_hash(self.conn),
            policy_hash=self.policy_hash,expires_at=time.time()+ttl_seconds,
            nonce=secrets.token_hex(16)
        )
        cap.signature=self._sign(cap.unsigned())
        return cap

    def commit_batch(self,cap,actor,operations,inject_failure_at=None):
        if not hmac.compare_digest(cap.signature,self._sign(cap.unsigned())):
            return {"committed":False,"reason":"BAD_SIGNATURE"}
        if cap.nonce in self.used_nonces:
            return {"committed":False,"reason":"REPLAY"}
        if time.time()>cap.expires_at:
            return {"committed":False,"reason":"EXPIRED"}
        if actor!=cap.actor or sha(operations)!=cap.operations_hash:
            return {"committed":False,"reason":"BINDING_MISMATCH"}
        if snapshot_hash(self.conn)!=cap.snapshot_hash:
            return {"committed":False,"reason":"STALE_SNAPSHOT"}
        if self.policy_hash!=cap.policy_hash:
            return {"committed":False,"reason":"POLICY_CHANGED"}

        predictions=[predict_archive(self.conn,op["args"]) for op in operations]
        predicted=sorted(set(i for p in predictions for i in p))
        before=current_status(self.conn)
        try:
            self.conn.execute("BEGIN IMMEDIATE")
            for idx,op in enumerate(operations):
                if inject_failure_at is not None and idx==inject_failure_at:
                    raise RuntimeError("synthetic provider failure")
                a=op["args"]
                self.conn.execute(
                    """UPDATE vendor_invoices SET status='ARCHIVED'
                       WHERE vendor_id=? AND month=? AND year=? AND status='OPEN'""",
                    (a["vendor_id"],a["month"],a["year"])
                )
            after=current_status(self.conn)
            actual=sorted(k for k,v in after.items() if before[k]!=v and v=="ARCHIVED")
            if actual!=predicted:
                self.conn.rollback()
                return {"committed":False,"reason":"ACTUAL_DELTA_MISMATCH"}
            self.conn.commit()
            self.used_nonces.add(cap.nonce)
            return {"committed":True,"reason":"OK"}
        except Exception:
            self.conn.rollback()
            return {"committed":False,"reason":"ROLLED_BACK"}

ops=[
    {"tool":"archive_invoices","args":{"vendor_id":4421,"month":3,"year":2025}},
    {"tool":"archive_invoices","args":{"vendor_id":4421,"month":4,"year":2025}},
]

conn=fresh_db(); broker=AtomicBatchBroker(conn)
cap=broker.issue_batch(ACTOR,ops)
ok=broker.commit_batch(cap,ACTOR,ops)
assert ok["committed"] is True

conn=fresh_db(); broker=AtomicBatchBroker(conn)
before=current_status(conn)
cap=broker.issue_batch(ACTOR,ops)
failed=broker.commit_batch(cap,ACTOR,ops,inject_failure_at=1)
after=current_status(conn)
assert failed["committed"] is False and before==after
print("Atomic rollback verified:",failed)

In [ ]:
attack_df.to_csv(RESULTS/"execution_attack_matrix.csv",index=False)
trials.to_csv(RESULTS/"execution_randomized_trials.csv",index=False)
metrics.to_csv(RESULTS/"execution_attack_summary.csv",index=False)
(RESULTS/"execution_security_metrics.json").write_text(json.dumps(security,indent=2))
(RESULTS/"execution_table.tex").write_text(metrics.to_latex(index=False,float_format=lambda x:f"{x:.4f}"))

module_text = (
    '"""Reference artifact generated by NTX_Q1_03_Verified_Execution_Broker.ipynb.\\n'
    'Integrate only after adapting the notebook implementation to project schemas.\\n'
    '"""\\n'
)
(RESULTS/"verified_execution_broker_reference.py").write_text(module_text)

fig,ax=plt.subplots(figsize=(8,4.5))
plot_df=metrics[metrics.attack!="valid"]
ax.bar(plot_df["attack"],1-plot_df["committed_rate"])
ax.set_ylim(0,1.02)
ax.set_ylabel("Blocked before commit")
ax.set_title("Verified Execution Broker attack blocking")
ax.tick_params(axis="x",rotation=40)
fig.tight_layout()
fig.savefig(RESULTS/"execution_attack_block_rate.png",dpi=220,bbox_inches="tight")
plt.show()

In [ ]:
import zipfile
zip_path=Path("/content/NTX_Q1_03_EXECUTION_BROKER_RESULTS.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.glob("execution_*"):
        z.write(p,arcname=p.name)
    p=RESULTS/"verified_execution_broker_reference.py"
    if p.exists():
        z.write(p,arcname=p.name)
print(zip_path)

In [ ]:
# DOWNLOAD RESULTS ZIP
from pathlib import Path
from google.colab import files

download_zip = Path("/content/NTX_Q1_03_EXECUTION_BROKER_RESULTS.zip")

if not download_zip.exists():
    raise FileNotFoundError(
        f"{download_zip} was not found. Run the result-export/ZIP cell above first."
    )

print(f"Downloading: {download_zip.name}")
files.download(str(download_zip))
